In [1]:
# Day 4 - Setup

!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq and LangChain connected successfully!")

Groq and LangChain connected successfully!


In [10]:
!pip install -q --upgrade --force-reinstall langchain langchain-core langchain-groq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-genai 2.12.1 requires tenacity<9.2.0,>=8.2.3, but you have tenacity 9.2.1 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.4 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 17.4 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.29.0 which is incompatible.


In [2]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("LLM initialized successfully!")

LLM initialized successfully!


In [8]:
!pip install -q langchain-groq langchain

In [1]:
# Google Sheets connection

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create the Orders spreadsheet
sh = gc.create("Sweet Crumbs Orders")
ws = sh.sheet1

# Add column headings
ws.append_row([
    "name",
    "item",
    "qty",
    "date",
    "eggless",
    "total"
])

print("Google Sheet connected successfully!")
print("Sheet URL:", sh.url)

Google Sheet connected successfully!
Sheet URL: https://docs.google.com/spreadsheets/d/1oWltybbl67f8yR7I4SPmsP9-_APVN3etJg3miSkwUM0


In [2]:
from langchain_core.tools import tool

MENU = {
    "chocolate cake": 650,
    "plum cake": 500
}

@tool
def get_price(cake_name: str) -> str:
    """Get the price of a cake from the Sweet Crumbs menu."""
    price = MENU.get(cake_name.lower())

    if price is None:
        return "Cake not found. Please check with the owner."

    return f"{cake_name}: Rs. {price}"

@tool
def is_open() -> str:
    """Check whether Sweet Crumbs bakery is currently open."""
    return "Business hours are not available. Please check with the owner."

print("Bakery tools created successfully!")
print(get_price.invoke({"cake_name": "Chocolate Cake"}))
print(is_open.invoke({}))

Bakery tools created successfully!
Chocolate Cake: Rs. 650
Business hours are not available. Please check with the owner.


In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class Order(BaseModel):
    name: str
    item: Literal["Chocolate Cake", "Plum Cake"]
    qty: int = Field(gt=0)
    date: str
    eggless: bool

print("Order model created successfully!")

sample_order = Order(
    name="Ravi",
    item="Chocolate Cake",
    qty=1,
    date="2026-11-28",
    eggless=True
)

print(sample_order)

Order model created successfully!
name='Ravi' item='Chocolate Cake' qty=1 date='2026-11-28' eggless=True


In [4]:
from langchain_core.tools import tool

@tool
def log_order(order: Order) -> str:
    """Validate and save a practice bakery order to Google Sheets."""
    try:
        validated_order = Order.model_validate(order)

        price = MENU[validated_order.item.lower()]
        total = price * validated_order.qty

        ws.append_row([
            validated_order.name,
            validated_order.item,
            validated_order.qty,
            validated_order.date,
            validated_order.eggless,
            total
        ])

        return f"Practice order recorded successfully! Total: Rs. {total}"

    except Exception as e:
        return f"Order could not be recorded: {e}"

print("log_order tool created successfully!")

log_order tool created successfully!


In [5]:
test_order = Order(
    name="Ravi",
    item="Chocolate Cake",
    qty=2,
    date="2026-11-28",
    eggless=True
)

result = log_order.invoke({
    "order": test_order.model_dump()
})

print(result)

Practice order recorded successfully! Total: Rs. 1300


In [7]:
from langchain.agents import create_agent

tools = [get_price, is_open, log_order]

system_prompt = """
You are the AI assistant for Sweet Crumbs Home Bakery.

Known menu:
- Chocolate Cake: Rs. 650
- Plum Cake: Rs. 500

Both cakes have eggless versions available.

Rules:
1. Use get_price to check cake prices.
2. Use is_open when asked about business hours.
3. Business hours are not confirmed. Never invent them.
4. Use log_order to record a practice order only after collecting
   customer name, cake name, quantity, required date, and eggless preference.
5. Never accept unknown cake names or quantities below 1.
6. Do not claim that a practice order is a real bakery booking.
7. Never invent delivery or advance-booking details.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Sweet Crumbs AI Agent created successfully!")

Sweet Crumbs AI Agent created successfully!


In [6]:
from langchain_core.tools import tool

MENU = {
    "chocolate cake": 650,
    "plum cake": 500
}

@tool
def get_price(cake_name: str) -> str:
    """Get the price of a cake."""
    price = MENU.get(cake_name.lower())

    if price is None:
        return "Cake not found. Please check with the owner."

    return f"{cake_name}: Rs. {price}"


@tool
def is_open() -> str:
    """Check bakery business hours."""
    return "Business hours are not confirmed. Please check with the owner."


@tool
def log_order(order: Order) -> str:
    """Record a validated practice order in Google Sheets."""
    validated_order = Order.model_validate(order)

    price = MENU[validated_order.item.lower()]
    total = price * validated_order.qty

    ws.append_row([
        validated_order.name,
        validated_order.item,
        validated_order.qty,
        validated_order.date,
        validated_order.eggless,
        total
    ])

    return f"Practice order recorded successfully! Total: Rs. {total}"


print("All three bakery tools recreated successfully!")

All three bakery tools recreated successfully!


In [5]:
from pydantic import BaseModel, Field
from typing import Literal

class Order(BaseModel):
    name: str
    item: Literal["Chocolate Cake", "Plum Cake"]
    qty: int = Field(gt=0)
    date: str
    eggless: bool

print("Order model recreated successfully!")

Order model recreated successfully!


In [8]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the price of a Plum Cake?"}
    ]
})

for message in response["messages"]:
    if message.type == "ai":
        print(message.content)


The price of a Plum Cake is **Rs. 500**.


In [9]:
test_questions = [
    "What is the price of a Chocolate Cake?",
    "What are your bakery opening hours?"
]

for question in test_questions:
    print("\nCustomer:", question)

    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    print("Assistant:", result["messages"][-1].content)


Customer: What is the price of a Chocolate Cake?
Assistant: The price of a Chocolate Cake is **Rs. 650**.

Customer: What are your bakery opening hours?
Assistant: I’m sorry, but the bakery’s opening hours are not confirmed at the moment. Please check with the owner for the most up‑to‑date schedule.


In [10]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Record a practice order for Ravi. "
                "Chocolate Cake, quantity 1, required date 2026-11-28, "
                "eggless true. This is only a practice record."
            )
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai":
        print(message.content)

NameError: name 'ws' is not defined

In [11]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open the existing spreadsheet
sh = gc.open_by_key("1oWltybbl67f8yR7I4SPmsP9-_APVN3etJg3miSkwUM0")
ws = sh.sheet1

print("Google Sheet reconnected successfully!")
print("Existing rows:", len(ws.get_all_values()))

Google Sheet reconnected successfully!
Existing rows: 2


In [12]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Record a practice order for Ravi. "
                "Chocolate Cake, quantity 1, required date 2026-11-28, "
                "eggless true. This is only a practice record."
            )
        }
    ]
})

for message in response["messages"]:
    if message.type == "ai":
        print(message.content)


✅ Practice order for Ravi has been recorded.  
- **Item:** Chocolate Cake (eggless)  
- **Quantity:** 1  
- **Required date:** 2026‑11‑28  
- **Total (practice):** Rs. 650  

This is a practice record only and does not constitute a real booking.


In [13]:
records = ws.get_all_records()

for record in records:
    print(record)

print("\nTotal practice orders:", len(records))

{'name': 'Ravi', 'item': 'Chocolate Cake', 'qty': 2, 'date': '2026-11-28', 'eggless': 'TRUE', 'total': 1300}
{'name': 'Ravi', 'item': 'Chocolate Cake', 'qty': 1, 'date': '2026-11-28', 'eggless': 'TRUE', 'total': 650}

Total practice orders: 2


In [14]:
for message in response["messages"]:
    print("Type:", message.type)
    print("Content:", message.content)

    if message.type == "ai":
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

Type: human
Content: Record a practice order for Ravi. Chocolate Cake, quantity 1, required date 2026-11-28, eggless true. This is only a practice record.
--------------------------------------------------
Type: ai
Content: 
Tool calls: [{'name': 'log_order', 'args': {'order': {'date': '2026-11-28', 'eggless': True, 'item': 'Chocolate Cake', 'name': 'Ravi', 'qty': 1}}, 'id': 'fc_ef4fc92c-6eeb-4b5e-86d4-c02ff964900f', 'type': 'tool_call'}]
--------------------------------------------------
Type: tool
Content: Practice order recorded successfully! Total: Rs. 650
--------------------------------------------------
Type: ai
Content: ✅ Practice order for Ravi has been recorded.  
- **Item:** Chocolate Cake (eggless)  
- **Quantity:** 1  
- **Required date:** 2026‑11‑28  
- **Total (practice):** Rs. 650  

This is a practice record only and does not constitute a real booking.
Tool calls: []
--------------------------------------------------


In [15]:
response_invalid = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Record a practice order for Ravi: Red Velvet Cake, quantity 1, date 2026-11-28, eggless true."
        }
    ]
})

print(response_invalid["messages"][-1].content)

I’m sorry, but we don’t offer a Red Velvet Cake on our menu. I can only take practice orders for the cakes we have—Chocolate Cake or Plum Cake. If you’d like to place an order for one of those, just let me know!


In [16]:
response_missing = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Record a practice order for Ravi: Chocolate Cake, quantity 1, eggless true. I haven't decided the required date yet."
        }
    ]
})

print(response_missing["messages"][-1].content)

Sure! To log the practice order, could you let me know the **required date** for the cake? Once I have that, I’ll record it for Ravi.


In [17]:
response_qty = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Record a practice order for Ravi: Chocolate Cake, quantity 0, required date 2026-11-28, eggless true."
        }
    ]
})

print(response_qty["messages"][-1].content)

I’m sorry, but the quantity must be at least 1. Could you please let me know how many cakes you’d like to order?


In [18]:
records = ws.get_all_records()

print("DAY 4 - FINAL SUMMARY")
print("-" * 35)
print("Total practice orders:", len(records))
print("Available cakes:", list(MENU.keys()))
print("Tools created:", [tool.name for tool in tools])
print("Agent status: Successfully created")
print("Google Sheets status: Connected")
print("Validation tests: 3/3 passed")

DAY 4 - FINAL SUMMARY
-----------------------------------
Total practice orders: 2
Available cakes: ['chocolate cake', 'plum cake']
Tools created: ['get_price', 'is_open', 'log_order']
Agent status: Successfully created
Google Sheets status: Connected
Validation tests: 3/3 passed
